# 02 · ETL e Integração SIH + CNES

**Objetivo:** Carregar os dados brutos do SIH e CNES, filtrar internações por IAM (CID I21),realizar limpeza e integrar as duas bases em uma base de modelagem unificada.

**Inputs:**
- `data/input/SIH/*.csv` — registros de AIH convertidos do SIH
- `data/input/CNES/*.csv` — tabelas do CNES (ST, LT, EQ, SR, HB)
- `data/external/dicionario_SIH.json` — mapeamento de colunas do SIH
- `data/external/dicionario_CNES_*.json` — mapeamentos de colunas do CNES

**Outputs gerados:**
- `data/interim/sih_iam.csv` — internações por IAM (base limpa SIH)
- `data/interim/cnes_hospitais.csv` — base mestre de hospitais (CNES consolidado)
- `data/processed/base_modelagem.csv` — base final SIH × CNES pronta para modelagem

## 0. Configuração do Ambiente

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import json
from pathlib import Path
import sys

In [2]:
# Adiciona a raiz do projeto ao sys.path para importar src/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: /home/carolina/Documents/TCC Documentos/TCC


In [3]:
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Caminhos ───────────────────────────────────────────────────────────
RAW_SIH  = Path(ROOT, 'data', 'input', 'SIH')
RAW_CNES = Path(ROOT, 'data', 'input', 'CNES')
INTERIM  = Path(ROOT, 'data', 'interim')
PROCESSED = Path(ROOT, 'data', 'processed')

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Caminhos configurados.")
print(f"  SIH  : {RAW_SIH}")
print(f"  CNES : {RAW_CNES}")

Caminhos configurados.
  SIH  : /home/carolina/Documents/TCC Documentos/TCC/data/input/SIH
  CNES : /home/carolina/Documents/TCC Documentos/TCC/data/input/CNES


## 1. Carregamento e Padronização do SIH

### 1.1 Leitura dos arquivos

Concatenamos **todos** os arquivos CSV disponíveis em `data/input/SIH/`.
O dicionário `dicionario_SIH.json` é usado para renomear as colunas para nomes legíveis.

In [4]:
arquivos_sih = sorted(RAW_SIH.glob('*.csv'))
print(f"Arquivos SIH encontrados: {len(arquivos_sih)}")
for f in arquivos_sih:
    print(f"  {f.name}")

Arquivos SIH encontrados: 12
  rdsp2501.csv
  rdsp2502.csv
  rdsp2503.csv
  rdsp2504.csv
  rdsp2505.csv
  rdsp2506.csv
  rdsp2507.csv
  rdsp2508.csv
  rdsp2509.csv
  rdsp2510.csv
  rdsp2511.csv
  rdsp2512.csv


In [5]:
# Carrega e renomeia colunas usando o dicionário externo
path_dicionario = Path(ROOT, 'data', 'external', 'dicionario_SIH.json')
with open(path_dicionario, 'r', encoding='utf-8') as f:
    schema = json.load(f)

rename_dict = {col["old_name"]: col["new_name"] for col in schema}

# Concatena todos os meses disponíveis
frames = []
for arq in arquivos_sih:
    df_tmp = pd.read_csv(arq, dtype=str, low_memory=False)
    df_tmp = df_tmp.rename(columns=rename_dict)
    frames.append(df_tmp)

df_sih = pd.concat(frames, ignore_index=True)
print(f"SIH carregado: {df_sih.shape[0]:,} registros × {df_sih.shape[1]} colunas")
df_sih.head(3)

SIH carregado: 2,948,801 registros × 114 colunas


,municipio_gestor,ano_competencia,mes_competencia,especialidade_leito,cnpj_hospital,numero_aih,tipo_aih,cep_paciente,municipio_residencia,data_nascimento,...,tipo_diag_sec_1,tipo_diag_sec_2,tipo_diag_sec_3,tipo_diag_sec_4,tipo_diag_sec_5,tipo_diag_sec_6,tipo_diag_sec_7,tipo_diag_sec_8,tipo_diag_sec_9,FONTE_ORC
0,350000,2025,01,01,46374500028366,3525100117847,1,11704840,354100,19840716,...,1,0,0,0,0,0,0,0,0,NaN
1,350000,2025,01,02,46374500028366,3524130275908,1,11741802,352210,20070606,...,1,1,1,1,1,0,0,0,0,NaN
2,350000,2025,01,02,46374500028366,3524130278427,1,11730000,353110,20030120,...,1,1,1,0,0,0,0,0,0,NaN


In [6]:
# Preview do SIH após renomeação de colunas
df_sih.head(3)

,municipio_gestor,ano_competencia,mes_competencia,especialidade_leito,cnpj_hospital,numero_aih,tipo_aih,cep_paciente,municipio_residencia,data_nascimento,...,tipo_diag_sec_1,tipo_diag_sec_2,tipo_diag_sec_3,tipo_diag_sec_4,tipo_diag_sec_5,tipo_diag_sec_6,tipo_diag_sec_7,tipo_diag_sec_8,tipo_diag_sec_9,FONTE_ORC
0,350000,2025,01,01,46374500028366,3525100117847,1,11704840,354100,19840716,...,1,0,0,0,0,0,0,0,0,NaN
1,350000,2025,01,02,46374500028366,3524130275908,1,11741802,352210,20070606,...,1,1,1,1,1,0,0,0,0,NaN
2,350000,2025,01,02,46374500028366,3524130278427,1,11730000,353110,20030120,...,1,1,1,0,0,0,0,0,0,NaN


### 1.2 Seleção de Colunas Relevantes

In [7]:
cols_keep = [
    "especialidade_leito", "numero_aih", "data_nascimento",
    "sexo", "uti_mes_total", "tipo_uti",
    "procedimento_solicitado", "procedimento_realizado",
    "data_internacao", "data_saida",
    "diagnostico_principal", "diagnostico_secundario",
    "motivo_saida", "codigo_idade", "idade", "dias_permanencia",
    "indicador_obito", "carater_internacao", "cid_notificacao",
    "cnes", "cid_associado", "cid_morte", "complexidade",
    "raca_cor", "etnia",
]

# Mantém apenas as colunas presentes no DataFrame (robustez)
cols_keep = [c for c in cols_keep if c in df_sih.columns]
df_sih = df_sih[cols_keep]
print(f"Colunas selecionadas: {len(cols_keep)}")
df_sih.head(3)

Colunas selecionadas: 25


,especialidade_leito,numero_aih,data_nascimento,sexo,uti_mes_total,tipo_uti,procedimento_solicitado,procedimento_realizado,data_internacao,data_saida,...,dias_permanencia,indicador_obito,carater_internacao,cid_notificacao,cnes,cid_associado,cid_morte,complexidade,raca_cor,etnia
0,01,3525100117847,19840716,3,0,00,0407040129,0407040129,20250116,20250117,...,1,0,01,NaN,2087804,0000,0000,02,01,0000
1,02,3524130275908,20070606,3,0,00,0310010039,0310010039,20241205,20241208,...,3,0,02,NaN,2087804,0000,0000,02,03,0000
2,02,3524130278427,20030120,3,0,00,0310010039,0310010039,20241212,20241216,...,4,0,02,NaN,2087804,0000,0000,02,01,0000


### 1.3 Filtro CID — Internações por IAM

Filtramos registros cujo diagnóstico principal **ou** secundário inicie com `I21`
(Infarto Agudo do Miocárdio — CID-10), que é o foco do estudo.

In [8]:
df_sih["diagnostico_principal"]  = df_sih["diagnostico_principal"].astype(str)
df_sih["diagnostico_secundario"] = df_sih["diagnostico_secundario"].astype(str)

df_iam = df_sih[
    df_sih["diagnostico_principal"].str.startswith("I21") |
    df_sih["diagnostico_secundario"].str.startswith("I21")
].copy()

print(f"Total de internações: {len(df_sih):,}")
print(f"Internações por IAM : {len(df_iam):,} ({len(df_iam)/len(df_sih)*100:.1f}%)")
df_iam.head(3)

Total de internações: 2,948,801
Internações por IAM : 49,047 (1.7%)


,especialidade_leito,numero_aih,data_nascimento,sexo,uti_mes_total,tipo_uti,procedimento_solicitado,procedimento_realizado,data_internacao,data_saida,...,dias_permanencia,indicador_obito,carater_internacao,cid_notificacao,cnes,cid_associado,cid_morte,complexidade,raca_cor,etnia
183,01,3524128896860,19740525,1,3,86,0406030049,0406030049,20241217,20241219,...,2,0,02,NaN,2077396,0000,0000,03,03,0000
184,01,3524128896871,19631015,3,0,00,0406030030,0406030030,20241211,20241212,...,1,0,02,NaN,2077396,0000,0000,03,03,0000
195,01,3524131709505,19550715,3,2,86,0406030022,0406030022,20241226,20241228,...,2,1,02,NaN,2077396,0000,0000,03,01,0000


### 1.4 Tratamento de Valores Ausentes

Campos administrativos do DataSUS frequentemente chegam como strings vazias ou `"0000"`.
Substituímos apenas **colunas de texto** (object), preservando zeros em variáveis
numéricas legítimas (ex.: `indicador_obito=0` = alta; `uti_mes_total=0` = sem UTI).

In [9]:
# Aplica replace somente em colunas de texto para não corromper variáveis numéricas/binárias
str_cols = df_iam.select_dtypes(include='str').columns
df_iam[str_cols] = df_iam[str_cols].replace(["", "0000", "000"], pd.NA)

# Converte colunas numéricas para tipo correto
num_cols = ["idade", "dias_permanencia", "indicador_obito",
            "uti_mes_total", "codigo_idade"]
for col in num_cols:
    if col in df_iam.columns:
        df_iam[col] = pd.to_numeric(df_iam[col], errors='coerce')

print("Tratamento de nulos concluído.")

Tratamento de nulos concluído.


In [10]:
# Resumo de completude
nulls = pd.DataFrame({
    "qtd_nulos":  df_iam.isnull().sum(),
    "perc_nulos": df_iam.isnull().mean() * 100
}).sort_values("perc_nulos", ascending=False)

nulls[nulls["qtd_nulos"] > 0]

,qtd_nulos,perc_nulos
diagnostico_secundario,49047,100.00
cid_notificacao,49047,100.00
cid_associado,49047,100.00
cid_morte,49047,100.00
etnia,49044,99.99


In [11]:
# ── Remoção de colunas inutilizáveis ─────────────────────────────────────────
# 1. Colunas 100% nulas (sem informação alguma)
cols_100pct_nulas = [c for c in df_iam.columns if df_iam[c].isnull().all()]
print(f'Colunas 100% nulas removidas ({len(cols_100pct_nulas)}): {cols_100pct_nulas}')
df_iam = df_iam.drop(columns=cols_100pct_nulas)

# 2. Colunas administrativas de álvara — irrelevantes para a modelagem
cols_alvara = [c for c in df_iam.columns if 'alvara' in c.lower()]
print(f'Colunas de álvara removidas ({len(cols_alvara)}): {cols_alvara}')
df_iam = df_iam.drop(columns=cols_alvara, errors='ignore')

# 3. Filtro de idade: manter apenas adultos (>= 18 anos)
#    IAM pediátrico é evento raro e biologicamente distinto — excluído do escopo
df_iam['idade'] = pd.to_numeric(df_iam['idade'], errors='coerce')
n_antes = len(df_iam)
df_iam = df_iam[df_iam['idade'] >= 18].copy()
print(f'Registros pediátricos removidos (idade < 18): {n_antes - len(df_iam)}')
print(f'Base SIH-IAM adultos: {len(df_iam):,} registros × {df_iam.shape[1]} colunas')

Colunas 100% nulas removidas (4): ['diagnostico_secundario', 'cid_notificacao', 'cid_associado', 'cid_morte']
Colunas de álvara removidas (0): []
Registros pediátricos removidos (idade < 18): 70
Base SIH-IAM adultos: 48,977 registros × 21 colunas


### 1.5 Salvar Base SIH-IAM Intermediária

In [12]:
path_sih_interim = INTERIM / "sih_iam.csv"
df_iam.to_csv(path_sih_interim, index=False)
print(f"SIH-IAM salvo em: {path_sih_interim}  ({len(df_iam):,} registros)")

SIH-IAM salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/sih_iam.csv  (48,977 registros)


## 2. Carregamento e Consolidação do CNES

Carregamos as cinco tabelas do CNES usando dicionários JSON específicos para cada prefixo.
A função `load_cnes_custom` lê o arquivo mais recente disponível por tabela.

In [13]:
def load_cnes_custom(prefix):
    """
    Carrega o arquivo CNES mais recente do prefixo informado,
    renomeia colunas via dicionário JSON e retorna o DataFrame filtrado.

    Parâmetros
    ----------
    prefix : str
        Prefixo da tabela CNES (ex.: 'st', 'lt', 'eq', 'sr', 'hb').

    Retorna
    -------
    pd.DataFrame — vazio se o dicionário ou os arquivos não forem encontrados.
    """
    path_dict = Path(ROOT, f'data/external/dicionario_CNES_{prefix.upper()}.json')
    if not path_dict.exists():
        print(f"  [AVISO] Dicionário não encontrado: {path_dict.name}")
        return pd.DataFrame()

    with open(path_dict, 'r', encoding='utf-8') as f:
        schema = json.load(f)

    rename_dict = {col["old_name"]: col["new_name"] for col in schema}
    cols_keep   = [col["new_name"] for col in schema]

    arquivos = sorted(RAW_CNES.glob(f'{prefix}sp*.csv'))
    if not arquivos:
        print(f"  [AVISO] Nenhum arquivo encontrado para prefixo '{prefix}'")
        return pd.DataFrame()

    # Usa o arquivo mais recente
    df = pd.read_csv(arquivos[-1], sep=',', encoding='latin-1',
                     dtype=str, on_bad_lines='skip')
    df = df.rename(columns=rename_dict)

    cols_presentes = [c for c in cols_keep if c in df.columns]
    return df[cols_presentes]


df_st = load_cnes_custom('st')
df_lt = load_cnes_custom('lt')
df_eq = load_cnes_custom('eq')
df_sr = load_cnes_custom('sr')
df_hb = load_cnes_custom('hb')

print(f"\nRegistros carregados por tabela:")
print(f"  ST (Estabelecimentos) : {len(df_st):>6,}")
print(f"  LT (Leitos)           : {len(df_lt):>6,}")
print(f"  EQ (Equipamentos)     : {len(df_eq):>6,}")
print(f"  SR (Serviços)         : {len(df_sr):>6,}")
print(f"  HB (Habilitações)     : {len(df_hb):>6,}")


Registros carregados por tabela:
  ST (Estabelecimentos) : 109,849
  LT (Leitos)           :  8,350
  EQ (Equipamentos)     : 242,363
  SR (Serviços)         : 177,050
  HB (Habilitações)     :  6,755


## 3. Consolidação das Tabelas CNES na Base ST

Cada extensão (LT, EQ, SR, HB) é agrupada por `codigo_cnes` e fundida
ao cadastro principal (ST) via *Left Join*.

### 3.1 Remoção de Duplicidades (ST)

In [14]:
# Garante uma linha por hospital — mantém o registro mais recente
n_antes = len(df_st)
df_st = df_st.drop_duplicates(subset=['codigo_cnes'], keep='last')
print(f"ST: {n_antes:,} → {len(df_st):,} registros únicos (removidas {n_antes - len(df_st):,} duplicatas)")

ST: 109,849 → 109,849 registros únicos (removidas 0 duplicatas)


### 3.2 Agregação de Leitos (LT)

In [15]:
# Leitos aparecem particionados por tipo; somamos por CNES
if not df_lt.empty:
    for col in ['quantidade_leitos_existentes', 'quantidade_leitos_sus',
                'quantidade_leitos_contratados']:
        if col in df_lt.columns:
            df_lt[col] = pd.to_numeric(df_lt[col], errors='coerce').fillna(0)

    df_lt_agg = df_lt.groupby('codigo_cnes', as_index=False).sum(numeric_only=True)
    df_st = pd.merge(df_st, df_lt_agg, on='codigo_cnes', how='left')
    print(f"Leitos agregados: {len(df_lt_agg):,} hospitais com dados de leitos.")
else:
    print("[AVISO] Tabela LT vazia — dados de leitos não incorporados.")

Leitos agregados: 1,462 hospitais com dados de leitos.


### 3.3 Agregação de Equipamentos (EQ)

In [16]:
# Equipamentos por instituição (soma de existentes e em uso)
if not df_eq.empty:
    for col in ['quantidade_existente', 'quantidade_em_uso']:
        if col in df_eq.columns:
            df_eq[col] = pd.to_numeric(df_eq[col], errors='coerce').fillna(0)

    df_eq_agg = df_eq.groupby('codigo_cnes', as_index=False).sum(numeric_only=True)
    df_st = pd.merge(df_st, df_eq_agg, on='codigo_cnes', how='left')
    print(f"Equipamentos agregados: {len(df_eq_agg):,} hospitais com dados de equipamentos.")
else:
    print("[AVISO] Tabela EQ vazia — dados de equipamentos não incorporados.")

Equipamentos agregados: 50,460 hospitais com dados de equipamentos.


### 3.4 Serviços Especializados (SR) e Habilitações (HB)

In [17]:
# SR e HB são categóricas — vincula o primeiro registro por CNES
if not df_sr.empty:
    df_sr_agg = df_sr.groupby('codigo_cnes', as_index=False).first()
    cols_dup  = [c for c in df_sr_agg.columns if c in df_st.columns and c != 'codigo_cnes']
    df_st = pd.merge(df_st, df_sr_agg.drop(columns=cols_dup), on='codigo_cnes', how='left')
    print(f"Serviços (SR) vinculados: {len(df_sr_agg):,} hospitais.")

if not df_hb.empty:
    df_hb_agg = df_hb.groupby('codigo_cnes', as_index=False).first()
    cols_dup  = [c for c in df_hb_agg.columns if c in df_st.columns and c != 'codigo_cnes']
    df_st = pd.merge(df_st, df_hb_agg.drop(columns=cols_dup), on='codigo_cnes', how='left')
    print(f"Habilitações (HB) vinculadas: {len(df_hb_agg):,} hospitais.")

Serviços (SR) vinculados: 41,011 hospitais.
Habilitações (HB) vinculadas: 2,282 hospitais.


### 3.5 Preenchimento de Nulos Numéricos pós-merge

Hospitais sem registros em LT/EQ terão `NaN` nos campos numéricos.
Preenchemos com `0` apenas colunas numéricas, o que é semanticamente
correto para contagens (0 leitos/equipamentos registrados).

In [18]:
df_cnes_final = df_st.copy()

numeric_cols = df_cnes_final.select_dtypes(include='number').columns
df_cnes_final[numeric_cols] = df_cnes_final[numeric_cols].fillna(0)

print(f"Base CNES consolidada: {df_cnes_final.shape[0]:,} hospitais × {df_cnes_final.shape[1]} colunas")
df_cnes_final.head(3)

Base CNES consolidada: 109,849 hospitais × 222 colunas


,codigo_cnes,codigo_municipio,cep_estabelecimento,cpf_cnpj_estabelecimento,tipo_pessoa,nivel_dependencia,cnpj_mantenedora,codigo_retencao_mantenedora,codigo_regiao_saude,codigo_micro_regiao_saude,...,indicador_terceirizado,caracterizacao_servico,indicador_servico_unico,codigo_cnes_terceiro,codigo_habilitacao,competencia_inicial,competencia_final,data_portaria,numero_portaria,competencia_portaria
0,0047406,350010,17800037,35723744000119,3,1,00000000000000,NaN,0209,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0081655,350010,17800057,00381929000108,3,1,00000000000000,NaN,R209,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0109789,350010,17803116,36060657000191,3,1,00000000000000,NaN,0209,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3.6 Remoção de Colunas com Excesso de Nulos (>50%)

In [19]:
# Identifica colunas com mais de 50% de valores ausentes
threshold = 0.50
cols_alta_nulidade = [
    c for c in df_cnes_final.columns
    if df_cnes_final[c].isnull().mean() > threshold
]

print(f"Colunas removidas por >50% nulos ({len(cols_alta_nulidade)}): {cols_alta_nulidade}")
df_cnes_final = df_cnes_final.drop(columns=cols_alta_nulidade)
print(f"Base após remoção: {df_cnes_final.shape[0]:,} hospitais × {df_cnes_final.shape[1]} colunas")

Colunas removidas por >50% nulos (37): ['codigo_retencao_mantenedora', 'codigo_regiao_saude', 'codigo_micro_regiao_saude', 'codigo_distrito_sanitario', 'codigo_modulo_assistencial', 'codigo_retencao_tributaria', 'codigo_natureza_organizacao', 'codigo_nivel_hierarquia', 'codigo_banco', 'codigo_agencia', 'conta_corrente', 'numero_contrato_municipal', 'data_publicacao_contrato_municipal', 'numero_contrato_estadual', 'data_publicacao_contrato_estadual', 'avaliado_acreditacao', 'classificacao_avaliacao', 'data_acreditacao', 'avaliado_pnass', 'data_avaliacao_pnass', 'codigo_servico_especializado', 'codigo_classificacao_servico', 'codigo_servico_unico', 'codigo_distrito_administrativo', 'indicador_pessoa', 'codigo_atividade_ensino', 'codigo_retencao_tributos', 'indicador_terceirizado', 'caracterizacao_servico', 'indicador_servico_unico', 'codigo_cnes_terceiro', 'codigo_habilitacao', 'competencia_inicial', 'competencia_final', 'data_portaria', 'numero_portaria', 'competencia_portaria']
Base ap

### 3.7 Verificação de Completude Final do CNES

In [20]:
nulos_cnes = pd.DataFrame({
    "qtd_nulos":  df_cnes_final.isnull().sum(),
    "perc_nulos": df_cnes_final.isnull().mean() * 100
}).sort_values("perc_nulos", ascending=False)

nulos_cnes[nulos_cnes["qtd_nulos"] > 0].head(20)

,qtd_nulos,perc_nulos
data_expedicao_alvara,23217,21.14
orgao_expedidor_alvara,22222,20.23
numero_alvara,21886,19.92
codigo_fluxo_clientela,941,0.86
codigo_turno_atendimento,111,0.10


### 3.8 Salvar Base Mestre de Hospitais (Interim CNES)

In [21]:
path_cnes_interim = INTERIM / "cnes_hospitais.csv"
df_cnes_final.to_csv(path_cnes_interim, index=False)
print(f"CNES Interim salvo em: {path_cnes_interim}  ({len(df_cnes_final):,} hospitais)")

CNES Interim salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/cnes_hospitais.csv  (109,849 hospitais)


## 4. Fusão Global: SIH-IAM × CNES

Fazemos um *Left Join* das internações por IAM com a base mestre de hospitais.
A chave de junção é `codigo_cnes`, normalizada em ambos os lados para evitar
divergências por zeros à esquerda ou espaços.

In [22]:
# Padroniza a chave de merge nos dois DataFrames
if 'cnes' in df_iam.columns:
    df_iam = df_iam.rename(columns={'cnes': 'codigo_cnes'})

df_iam['codigo_cnes']        = df_iam['codigo_cnes'].astype(str).str.strip().str.zfill(7)
df_cnes_final['codigo_cnes'] = df_cnes_final['codigo_cnes'].astype(str).str.strip().str.zfill(7)

# Left Join: mantém todos os pacientes IAM
df_base_modelagem = pd.merge(df_iam, df_cnes_final, on='codigo_cnes', how='left')

# Log de match rate
n_sem_cnes = df_base_modelagem[df_cnes_final.columns.difference(['codigo_cnes'])[0]].isna().sum()
print(f"Total de internações IAM          : {len(df_base_modelagem):,}")
print(f"Sem correspondência no CNES       : {n_sem_cnes:,} ({n_sem_cnes/len(df_base_modelagem)*100:.1f}%)")
print(f"Com dados hospitalares do CNES    : {len(df_base_modelagem) - n_sem_cnes:,}")
print(f"Número de features da base final  : {df_base_modelagem.shape[1]}")

Total de internações IAM          : 48,977
Sem correspondência no CNES       : 1 (0.0%)
Com dados hospitalares do CNES    : 48,976
Número de features da base final  : 205


In [23]:
# Exporta base de modelagem final
path_base_final = PROCESSED / "base_modelagem.csv"
df_base_modelagem.to_csv(path_base_final, index=False)
print(f"Base de modelagem salva em: {path_base_final}")
df_base_modelagem.sample(3)

Base de modelagem salva em: /home/carolina/Documents/TCC Documentos/TCC/data/processed/base_modelagem.csv


,especialidade_leito,numero_aih,data_nascimento,sexo,uti_mes_total,tipo_uti,procedimento_solicitado,procedimento_realizado,data_internacao,data_saida,...,regulacao_convenio_particular,regulacao_plano_publico,regulacao_plano_privado,regulacao_seguro_proprio,regulacao_seguro_terceiro,quantidade_leitos_existentes,quantidade_leitos_contratados,quantidade_leitos_sus,quantidade_existente,quantidade_em_uso
38440,01,3525108162301,19570304,1,0,00,0406030030,0406030030,20250929,20251001,...,0,0,0,0,0,1305.00,0.00,874.00,4384.00,4352.00
40656,01,3525128021272,19480506,1,5,75,0406030049,0406030049,20251025,20251101,...,0,0,0,0,0,249.00,0.00,213.00,1003.00,998.00
20113,03,3525102781849,19941112,1,0,00,0303060190,0303060190,20250516,20250528,...,0,0,0,0,0,295.00,0.00,295.00,1177.00,1175.00


In [25]:
df_base_modelagem

,especialidade_leito,numero_aih,data_nascimento,sexo,uti_mes_total,tipo_uti,procedimento_solicitado,procedimento_realizado,data_internacao,data_saida,...,regulacao_convenio_particular,regulacao_plano_publico,regulacao_plano_privado,regulacao_seguro_proprio,regulacao_seguro_terceiro,quantidade_leitos_existentes,quantidade_leitos_contratados,quantidade_leitos_sus,quantidade_existente,quantidade_em_uso
0,01,3524128896860,19740525,1,3,86,0406030049,0406030049,20241217,20241219,...,0,0,0,0,0,1305.00,0.00,874.00,4384.00,4352.00
1,01,3524128896871,19631015,3,0,00,0406030030,0406030030,20241211,20241212,...,0,0,0,0,0,1305.00,0.00,874.00,4384.00,4352.00
2,01,3524131709505,19550715,3,2,86,0406030022,0406030022,20241226,20241228,...,0,0,0,0,0,1305.00,0.00,874.00,4384.00,4352.00
3,01,3524128891019,19510501,3,2,86,0406030049,0406030049,20241212,20241215,...,0,0,0,0,0,1305.00,0.00,874.00,4384.00,4352.00
4,01,3524128891437,19600521,1,5,86,0406030049,0406030049,20241213,20241217,...,0,0,0,0,0,1305.00,0.00,874.00,4384.00,4352.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48972,03,3525111826830,19681205,1,0,00,0303060190,0303060190,20251124,20251201,...,0,0,0,0,0,34.00,0.00,32.00,27.00,27.00
48973,03,3525123495950,19770325,1,0,00,0303060190,0303060190,20251129,20251205,...,0,0,0,0,0,100.00,0.00,63.00,165.00,165.00
48974,03,3525123494981,19610613,1,0,00,0303060190,0303060190,20251215,20251225,...,0,0,0,0,0,100.00,0.00,63.00,165.00,165.00
48975,03,3525123495091,19460204,1,0,00,0303060190,0303060190,20251211,20251223,...,0,0,0,0,0,100.00,0.00,63.00,165.00,165.00


---
## Resumo do Pipeline

| Etapa | Input | Output | Registros |
|-------|-------|--------|-----------|
| 1. Filtro IAM (SIH) | `data/input/SIH/*.csv` | `data/interim/sih_iam.csv` | ver acima |
| 2. Consolidação CNES | `data/input/CNES/*.csv` | `data/interim/cnes_hospitais.csv` | ver acima |
| 3. Fusão global | sih_iam + cnes_hospitais | `data/processed/base_modelagem.csv` | ver acima |

> **Próximos passos:** notebook `03_analise_exploratotia_visualizacao.ipynb` — análise exploratória e visualizações.